In [ ]:
%load_ext manim

# Simulación del lanzamiento de 10 dados con Manim

En este cuaderno se desarrolla una simulación visual del lanzamiento simultáneo de diez dados.

## Objetivo
Representar de forma animada un experimento aleatorio discreto en el que, para cada tirada:

- se generan 10 variables aleatorias con valores entre 1 y 6,
- se visualiza cada dado individualmente,
- se calcula la suma total obtenida,
- y se actualiza dinámicamente el resultado en pantalla.

In [ ]:
from manim import *
import random

## Representación de un dado

Cada dado está compuesto por:
- una cara con esquinas redondeadas,
- un relleno gris para mejorar la estética,
- y una configuración de puntos que depende del valor mostrado.

El uso de 'VGroup' permite tratar todos esos elementos como un único objeto animable.

In [ ]:
# Creamos una clase para representar el dado
class Dado(VGroup): 
    def __init__(self, valor=1, size=1): # Valor del dado (1-6) y tamaño
        super().__init__() # Comando para la inicialización del grupo 
        self.size = size # Guarda el tamaño del dado para usarlo después
        
        # Creación del cuadrado con bordes redondeados con las caracteristicas elejidas
        square = RoundedRectangle(
            corner_radius=0.2*size,
            height=size, 
            width=size, 
            stroke_width=2, 
            color=WHITE, 
            fill_color=GREY_E, 
            fill_opacity=1 
        )

        s = size * 0.3 # Distancia entre puntos
        # Posiciones de los puntos para cada valor del dado
        posiciones = {
            1: [(0, 0)],
            2: [(-s, -s), (s, s)], 
            3: [(-s, -s), (0, 0), (s, s)],
            4: [(-s, -s), (-s, s), (s, -s), (s, s)],
            5: [(-s, -s), (-s, s), (0, 0), (s, -s), (s, s)],
            6: [(-s, -s), (-s, 0), (-s, s),
                (s, -s), (s, 0), (s, s)]
        }

        # Creamos los puntos del dado según el valor
        puntos = VGroup(*[
            Dot( 
                [x, y, 0], #Posición
                radius=0.08*size, #Tamaño del punto
                color=YELLOW #Color
            )
            for x, y in posiciones[valor] 
        ])

        self.add(square, puntos) # Agregamos el cuadrado y los puntos al grupo del dado

## Escena principal: simulación de lanzamientos

Una vez definido el objeto 'Dado', se construye una escena que simula tiradas sucesivas.

En cada iteración:
- Se generan nuevos valores aleatorios.
- Cada dado se transforma visualmente en su nuevo resultado.
- Se calcula la suma de los diez valores.
- El texto se actualiza mostrando el nuevo resultado.

La animación utiliza transformaciones para reproducir visualmente el proceso de lanzar los dados.

In [ ]:
%%manim -ql Dados

# Creamos la escena principal donde se mostrarán los dados y la suma
class Dados(Scene):
    def construct(self): 
        n = 10 # Número de dados

        # Creamos un grupo de dados con valores aleatorios entre 1 y 6 haceiendo referencia a la clase 'Dado' creada anteriormente
        dados = VGroup(*[
            Dado(random.randint(1, 6), size=1) # Crea un dado con valor aleatorio y tamaño 1
            for _ in range(n) # Crea los n dados seleccionados
        ])

        dados.arrange_in_grid(rows=2, cols=5, buff=0.4) # Organiza los dados en una cuadrícula de 2 filas y 5 columnas con espacio entre ellos
        dados.move_to(ORIGIN + UP*0.8)   # Mueve los dados hacia arriba

        texto = Text("Suma: 0", font="Arial", color=YELLOW).scale(0.8) # Crea un texto para mostrar la suma de los dados
        texto.next_to(dados, DOWN, buff=0.6)  # Posiciona el texto debajo de los dados

        self.play(FadeIn(dados), FadeIn(texto)) # Anima la aparición de los dados y el texto

        # Simulamos el lanzamiento de los dados 10 veces
        for _ in range(10):
            nuevos = []
            valores = []

            # Para cada dado, generamos un nuevo valor aleatorio y creamos un nuevo dado con ese valor
            for d in dados:
                val = random.randint(1, 6)
                valores.append(val)

                nuevo = Dado(val, size=1)
                nuevo.move_to(d.get_center())
                nuevos.append(nuevo)

            # Suma los valores de los n dados
            suma = sum(valores)

            # Anima la trasnsición en el acmbio de dados
            self.play(
                *[Transform(d, n) for d, n in zip(dados, nuevos)],
                run_time=0.4
            )

            # Actualiza el texto con la nueva suma
            nuevo_texto = Text(f"Suma: {suma}", font="Arial", color=YELLOW).scale(0.9)
            nuevo_texto.move_to(texto)
            self.play(Transform(texto, nuevo_texto), run_time=0.3)

            self.wait(0.3)

        self.wait(2)